
#### Shared preprocessing for both ML and DL

In [1]:
import os
import json
import re
import string
import pandas as pd
from sklearn.model_selection import train_test_split

### Config

In [2]:
RAW_FILE = "../data/raw/all_emotions-2.csv"
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE = "../data/splits/shared_split.json"

RANDOM_STATE = 42
NEUTRAL_LIMIT = 500
TEST_SIZE = 0.15
VAL_SIZE = 0.15

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/splits", exist_ok=True)

### Clean text

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


### Load dataset

In [4]:
df = pd.read_csv(RAW_FILE)
print("Original shape:", df.shape)

Original shape: (9021, 3)


### Filter only patient

In [5]:
df.columns = df.columns.str.strip().str.lower()
df["speaker"] = df["speaker"].astype(str).str.lower().str.strip()
df = df[df["speaker"] == "patient"].copy()

print("After filtering patient:", df.shape)

After filtering patient: (4516, 3)


### Keep needed columns

In [6]:
df = df[["utterance", "emotion"]].copy()
df.dropna(subset=["utterance", "emotion"], inplace=True)

df["utterance"] = df["utterance"].astype(str)
df["emotion"] = df["emotion"].astype(str).str.strip()

df = df[df["utterance"].str.strip() != ""]
df = df[df["emotion"].str.strip() != ""]

### Clean text

In [7]:
df["text"] = df["utterance"].apply(clean_text)
df = df[df["text"].str.strip() != ""].copy()

### Limit neutral

In [8]:
neutral_df = df[df["emotion"].str.lower() == "neutral"].copy()
other_df = df[df["emotion"].str.lower() != "neutral"].copy()

if len(neutral_df) > NEUTRAL_LIMIT:
    neutral_df = neutral_df.sample(n=NEUTRAL_LIMIT, random_state=RANDOM_STATE)

df = pd.concat([neutral_df, other_df], axis=0)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

### Final shared dataset

In [9]:
df = df[["text", "emotion"]].copy()
df.columns = ["text", "label"]
df["row_id"] = range(len(df))

print("\nFinal shared dataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())

df.to_csv(SHARED_DATASET_FILE, index=False, encoding="utf-8-sig")
print("\nSaved shared dataset:", SHARED_DATASET_FILE)


Final shared dataset shape: (2160, 3)

Label distribution:
label
neutral    500
disgust    427
fear       395
angry      326
sad        278
happy      234
Name: count, dtype: int64

Saved shared dataset: ../data/processed/cleaned_emotions_shared.csv


### Shared split

In [10]:
train_df, temp_df = train_test_split(
    df,
    test_size=(TEST_SIZE + VAL_SIZE),
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

relative_test_size = TEST_SIZE / (TEST_SIZE + VAL_SIZE)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_size,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

split_dict = {
    "train_ids": train_df["row_id"].tolist(),
    "val_ids": val_df["row_id"].tolist(),
    "test_ids": test_df["row_id"].tolist()
}

with open(SHARED_SPLIT_FILE, "w", encoding="utf-8") as f:
    json.dump(split_dict, f, ensure_ascii=False, indent=2)

print("\nSplit sizes")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nSaved shared split:", SHARED_SPLIT_FILE)


Split sizes
Train: (1512, 3)
Val  : (324, 3)
Test : (324, 3)

Saved shared split: ../data/splits/shared_split.json
